# 04 — Interactive Dashboard

**Objective**: generate the inference dashboard (light theme, orange accents, embedding scatter, similarity heatmap, score distribution -- see the Task 1 dashboard refactor) from a set of predictions + item embeddings, and view it inline in this notebook.

**Audience**: anyone who wants to sanity-check a model's recommendations visually instead of reading raw numbers.

This notebook does the **real** thing (`src.models.arycolbring.narative.rearender.generate_inference_report`) rather than a mock -- the dashboard code itself is pure Python/Jinja2, so as long as your environment has the project's normal dependencies installed (`duckdb`, `tqdm`, `jinja2` -- all in `requirements.txt`), this runs end to end.

## 1. Setup + a few predictions to visualize

In [ ]:
from pathlib import Path
import numpy as np

np.random.seed(0)
OUTPUT_DIR = Path("./notebook_dashboard_output")

N_ITEMS, DIM = 60, 16
item_embeddings = np.random.default_rng(0).normal(size=(N_ITEMS, DIM))
item_ids = list(range(N_ITEMS))

# A small set of predictions for one user (in a real run, this comes from
# AryColBringInference.recommend() -- see 01_Quick_Start.ipynb).
scores = item_embeddings @ np.random.default_rng(1).normal(size=DIM)
top10 = np.argsort(scores)[::-1][:10]
predictions = [{"user_id": 0, "item_id": int(i), "score": float(scores[i]), "rank": r + 1}
               for r, i in enumerate(top10)]
for p in predictions[:3]:
    print(p)

## 2. Generate the dashboard

In [ ]:
try:
    from src.models.arycolbring.narative.rearender import generate_inference_report

    context_data = {
        "metrics": {"avg_latency_ms": 9.4, "qps": 780, "throughput_preds_per_sec": 610.0},
        "inference_statistics": {"n_predictions": len(predictions), "n_users_served": 1,
                                 "avg_latency_ms": 9.4, "throughput_preds_per_sec": 610.0},
        "experiment_name": "Notebook Demo Run",
        "predictions": predictions,
        "item_embeddings": item_embeddings,
        "item_ids": item_ids,
        "embeddings": {"vectors": item_embeddings, "ids": item_ids, "highlight_id": top10[0]},
    }
    report_path = generate_inference_report(context_data, output_dir=OUTPUT_DIR)
    print(f"Report written to: {report_path}")
    DASHBOARD_AVAILABLE = True
except ImportError as exc:
    print(f"Could not import the dashboard module in this environment: {exc}")
    print("Install the project's normal dependencies (see requirements.txt) to run this cell for real.")
    report_path = None
    DASHBOARD_AVAILABLE = False

## 3. View it inline

`IFrame` embeds the generated HTML report (with its own CSS/JS) directly in the notebook output.

In [ ]:
if DASHBOARD_AVAILABLE:
    from IPython.display import IFrame
    display(IFrame(src=str(report_path), width="100%", height=650))
else:
    print("Skipped: see the note above.")

## 4. What you should see

- **Overview** tab: scorecards for `avg_latency_ms` / `qps` / `throughput_preds_per_sec` only -- no fabricated "coverage" number, no eval-quality gauges (Precision/Recall/NDCG/AUC/MRR don't belong on a *production inference* report; see the training dashboard for those).
- **Rankings** tab: the 10 predictions, sortable by score, each with a "Similar Items (Top 3)" column computed from cosine similarity over `item_embeddings`.
- **Insights** tab (new): prediction score histogram, a 2D PCA projection of `item_embeddings` (the recommended item highlighted), and an item-item similarity heatmap.

## Summary

- `generate_inference_report(context_data, output_dir=...)` is the one call needed to go from predictions + embeddings to a full HTML dashboard.
- The dashboard needs no Cython/compiled dependency -- only the project's normal Python dependencies.

**Next**: `05_LTR_LGBM_Comparison.ipynb` to compare AryColBring's recommendations against the LTR-LGBM model.